# Stage 4: Socket Detection (Downsampled)

Detect basal socket using 3D convex hull and morphological operations.
All data is already downsampled from Stage 0 for ~8x speedup.

**Input**: `labelsShrunk_95_ds/*_shrunk.nii.gz`  
**Output**: `labelsSocketDetected_ds/*_socket.nii.gz`  
**Metrics**: `metrics/socket_metrics_ds.json`

In [ ]:
# Configuration
TARGET_DIR = "/mnt/c/users/mwild/firebase/perios/levi_data_1.6.26"

# Morphological parameters (adjusted for downsampled resolution)
# Original was 10 at full res, so 5 at half res
EROSION_ITERATIONS = 5
DILATION_ITERATIONS = 5

In [ ]:
import sys
from pathlib import Path
import numpy as np
from scipy import ndimage

sys.path.insert(0, str(Path('.').resolve()))
from utils import (
    load_nifti, save_nifti, save_metrics, ensure_dir,
    largest_component, compute_3d_convex_hull, compute_socket_metrics
)

In [ ]:
# Setup directories (all downsampled with _ds suffix)
target = Path(TARGET_DIR)
INPUT_DIR = target / "labelsShrunk_95_ds"
METRICS_DIR = ensure_dir(target / "metrics")

# Create all output directories (downsampled)
OUTPUT_DIRS = {
    'hull': ensure_dir(target / "labelsConvexHull3D_ds"),
    'difference': ensure_dir(target / "labelsHullDifference_ds"),
    'eroded': ensure_dir(target / "labelsHullDiff_eroded_ds"),
    'dilated': ensure_dir(target / "labelsHullDiff_dilated_ds"),
    'socket': ensure_dir(target / "labelsSocketDetected_ds"),
}

print(f"Input: {INPUT_DIR}")
print(f"Output directories (downsampled):")
for name, path in OUTPUT_DIRS.items():
    print(f"  {name}: {path}")

In [ ]:
def process_socket(shrunk_file, output_dirs):
    """Process a single sample through the socket detection pipeline."""
    sample_name = shrunk_file.stem.replace('_shrunk', '').replace('.nii', '')
    
    # Load shrunk mask
    shrunk_data, affine, header = load_nifti(shrunk_file)
    shrunk_mask = (shrunk_data > 0).astype(np.uint8)
    
    # Step 1: Compute 3D convex hull
    print(f"    Computing 3D convex hull...")
    hull_mask = compute_3d_convex_hull(shrunk_mask)
    hull_output = output_dirs['hull'] / f"{sample_name}_hull3d.nii.gz"
    save_nifti(hull_mask, affine, header, hull_output)
    
    # Step 2: Compute difference
    difference = (hull_mask > 0) & (shrunk_mask == 0)
    difference = difference.astype(np.uint8)
    diff_output = output_dirs['difference'] / f"{sample_name}_difference.nii.gz"
    save_nifti(difference, affine, header, diff_output)
    
    # Step 3: Erode
    print(f"    Eroding by {EROSION_ITERATIONS} voxels...")
    eroded = ndimage.binary_erosion(difference, iterations=EROSION_ITERATIONS).astype(np.uint8)
    eroded_output = output_dirs['eroded'] / f"{sample_name}_eroded.nii.gz"
    save_nifti(eroded, affine, header, eroded_output)
    
    # Step 4: Dilate
    print(f"    Dilating by {DILATION_ITERATIONS} voxels...")
    dilated = ndimage.binary_dilation(eroded, iterations=DILATION_ITERATIONS).astype(np.uint8)
    dilated_output = output_dirs['dilated'] / f"{sample_name}_dilated.nii.gz"
    save_nifti(dilated, affine, header, dilated_output)
    
    # Step 5: Largest connected component
    labeled, num_features = ndimage.label(dilated)
    if num_features > 0:
        socket_mask = largest_component(dilated)
    else:
        socket_mask = np.zeros_like(dilated)
    
    socket_output = output_dirs['socket'] / f"{sample_name}_socket.nii.gz"
    save_nifti(socket_mask, affine, header, socket_output)
    
    # Compute metrics (in downsampled space - will be scaled in upsample stage)
    metrics = compute_socket_metrics(shrunk_mask, hull_mask, socket_mask, num_features)
    metrics['difference_volume'] = int(np.sum(difference))
    metrics['eroded_volume'] = int(np.sum(eroded))
    metrics['dilated_volume'] = int(np.sum(dilated))
    
    return metrics

In [ ]:
# Process all shrunk masks
shrunk_files = sorted(INPUT_DIR.glob("*_shrunk.nii.gz"))
print(f"Found {len(shrunk_files)} masks to process")
print(f"Processing on downsampled data (~8x speedup)")
print("="*70)

all_metrics = []

import time
total_start = time.time()

for idx, shrunk_file in enumerate(shrunk_files, 1):
    sample_name = shrunk_file.stem.replace('_shrunk', '').replace('.nii', '')
    print(f"\n[{idx}/{len(shrunk_files)}] {sample_name}")
    
    start_time = time.time()
    
    try:
        metrics = process_socket(shrunk_file, OUTPUT_DIRS)
        metrics['filename'] = shrunk_file.name
        metrics['sample_name'] = sample_name
        all_metrics.append(metrics)
        
        elapsed = time.time() - start_time
        print(f"    Socket volume: {metrics['socket_volume']:,} voxels (downsampled)")
        if metrics['equivalent_radius']:
            print(f"    Equivalent radius: {metrics['equivalent_radius']:.2f} voxels (downsampled)")
        print(f"    Time: {elapsed:.1f}s")
    except Exception as e:
        print(f"    ERROR: {e}")
        import traceback
        traceback.print_exc()

total_elapsed = time.time() - total_start
print(f"\nTotal processing time: {total_elapsed/60:.1f} minutes")

In [ ]:
# Save metrics
metrics_file = METRICS_DIR / "socket_metrics_ds.json"
save_metrics(all_metrics, metrics_file)

# Summary
print("\n" + "="*70)
print("SOCKET DETECTION COMPLETE (DOWNSAMPLED)")
print("="*70)
print(f"\nProcessed: {len(all_metrics)}/{len(shrunk_files)} samples")

if all_metrics:
    socket_vols = [m['socket_volume'] for m in all_metrics if m['socket_volume'] > 0]
    radii = [m['equivalent_radius'] for m in all_metrics if m['equivalent_radius']]
    
    if socket_vols:
        print(f"\nSocket metrics (downsampled space):")
        print(f"  Volume: {min(socket_vols):,} - {max(socket_vols):,} voxels")
    if radii:
        print(f"  Radius: {min(radii):.2f} - {max(radii):.2f} voxels")

print(f"\nOutput: {OUTPUT_DIRS['socket']}")
print(f"Metrics: {metrics_file}")
print(f"\nNote: Metrics will be scaled in the final upsample stage")